# 02 · Topics: what are the calls about? (Task 1)

**Transcript Intelligence: notebook 2 of 5**

**The task.** Build a pipeline that categorizes calls by topic or theme, explain why the approach
was chosen, and give examples from each category.

**The answer in one line.** A **hybrid** pipeline (`src/topics.py`): unsupervised clustering
*discovers* the structure, a fixed 10-theme taxonomy *defines* it in business language, Claude
*classifies* each call (one primary theme plus every theme that applies), and keyword rules provide
both a fallback and an independent check.

**How to run.** `python -m src.pipeline` builds two versions of the topic outputs:
`data/processed/hybrid/` (Claude + rules) and `data/processed/rules/` (keyword rules only, no API
calls). This notebook reads both and compares them. It makes no LLM calls.

## 1. Setup

In [ ]:
%load_ext autoreload
%autoreload 2

import sys, json
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.io as pio

from src.load import PROCESSED_DIR, mode_dir
from src.enrich import load_enriched, PERIODS
from src.topics import THEMES, THEME_NAMES, SYSTEM_PROMPT, N_CLUSTERS, EMBED_MODEL, load_topics, call_text, embed_calls
from src.llm import load_usage, uncosted_cache_entries

pio.renderers.default = "plotly_mimetype+notebook_connected"
pd.set_option("display.max_columns", 50)
pd.set_option("display.max_colwidth", 140)
TYPE_COLORS = {"external": "#2a78d6", "support": "#eb6834", "internal": "#8a8f98"}

In [ ]:
calls = load_enriched(PROCESSED_DIR)["calls"]
ORIGINAL_END = pd.Timestamp("2026-04-28").date()
calls["is_synthetic"] = calls["date"].map(lambda d: pd.Timestamp(d).date() > ORIGINAL_END)

hyb, clusters, hyb_metrics = load_topics(mode_dir("hybrid"))
rul, _, rul_metrics = load_topics(mode_dir("rules"))

meta = ["call_id", "date", "title", "call_type", "account", "period", "sentiment_score", "is_synthetic"]
H = hyb.merge(calls[meta], on="call_id")
R = rul.merge(calls[meta], on="call_id")
Ho = H[~H["is_synthetic"]]          # original 100 calls, used for headline numbers
print(f"{len(H)} calls classified ({len(Ho)} original, {H['is_synthetic'].sum()} synthetic); "
      f"hybrid sources: {hyb['source'].value_counts().to_dict()}; model: {hyb_metrics.get('model')}")

## 2. Why not just use the provided topics?

Each call already has free-text `topics` from the transcription vendor. `01_explore` showed they
can't be used directly: 351 distinct labels across 100 calls, most used only once, and one concept
split many ways ("outage", "platform outage", "service outage", "incident response"…). A
stakeholder can't filter or trend on that. We need a **small, stable, readable** set of categories.

### Options considered

| Approach | Strength | Weakness | Role here |
|---|---|---|---|
| **Keyword rules** | Free, instant, fully transparent | Brittle: matches words, not meaning; misses context | Fallback + independent check |
| **Clustering (embeddings)** | Finds structure without assumptions | Clusters aren't named, can form around noise (a customer's name) | **Discover** the themes |
| **LLM classification** | Understands context, handles calls with several themes, explains itself | Costs money; non-deterministic unless controlled | **Classify** each call |
| **Hybrid** (chosen) | Data-grounded taxonomy + nuanced labels + a built-in cross-check | More moving parts | ✓ |

**Decision.** Use each method for what it's good at, and use the rules to *check* the LLM rather
than trust it blindly.

## 3. Step 1: Discover (clustering)

Each call's title, summary and vendor topics are embedded with a local sentence-embedding model
(`all-MiniLM-L6-v2`, free, no API) and grouped with KMeans into 10 clusters. Each cluster is
described by its most distinctive TF-IDF terms.

In [ ]:
clusters.sort_values("n_calls", ascending=False)[["cluster", "n_calls", "top_terms", "dominant_theme", "purity"]]

In [ ]:
# 2-D map of the clusters (PCA of the same embeddings used for clustering)
from sklearn.decomposition import PCA
texts = calls.set_index("call_id").loc[H["call_id"]].reset_index().apply(call_text, axis=1).tolist()
xy = PCA(n_components=2, random_state=0).fit_transform(embed_calls(texts))
plot = H.assign(x=xy[:, 0], y=xy[:, 1], cluster=H["cluster"].astype(str))
px.scatter(plot, x="x", y="y", color="primary_theme", symbol="call_type", hover_data=["title", "cluster"],
           title="Calls in embedding space, coloured by final primary theme").update_traces(marker_size=9)

**What this shows.** Most clusters map onto a recognisable theme: the outage (two clusters), Comply v2,
renewals, billing, backup, Identity and sprint planning. That confirms these themes are real in the data,
not invented.

It also shows why clustering **can't be the final answer**:

- **Some clusters form partly around people, not topics.** In the current run one cluster's top
  terms include a customer name ("silverline"), and two others include a staff member's first
  name ("maria", "marcus"). KMeans groups whatever is most similar, and repeated names are very
  similar. The exact terms shift a little when new transcripts are added.
- **Each call gets exactly one cluster**, but many calls are about several things. A renewal call
  dominated by the outage belongs in both groups.
- **Clusters have no names.** Someone has to turn "renewal, aegis, pricing, outage, maria" into a
  category a sales leader would use.

## 4. Step 2: Define the taxonomy

The 10 themes were consolidated from the clusters above, the vendor's 351 labels, and the call
types. Some business-critical themes, such as **Billing disputes** and **Competitive threat**, are
too small or too spread out to form their own cluster, but a stakeholder still needs them as
categories. Each theme has a plain-English **definition** (what the LLM sees) and a **keyword rule**
(the fallback and check).

In [ ]:
pd.DataFrame([{"theme": n, "definition": d, "keyword rule": p.pattern} for n, (d, p) in THEMES.items()])

## 5. Step 3: Classify each call with Claude

For each call, Claude receives the title, call type, date, summary, vendor topics and key moments,
and returns structured JSON validated by a pydantic schema:

| Field | Meaning |
|---|---|
| `primary_theme` | the single theme the call is mostly about |
| `themes` | every theme that is a substantive part of the call (multi-label) |
| `certainty` | confidence in the primary theme, 0–1 |
| `rationale` | one sentence citing the evidence |

**Engineering choices**

- **Constrained output.** Themes must come from the taxonomy, so the model can't invent a category.
- **Cached.** Each answer is stored in `data/processed/llm_cache/`, keyed by model, prompt version
  and input. Re-runs are free and give identical results. Bump `PROMPT_VERSION` to invalidate.
- **Fails soft.** If the API is unavailable or the model declines, keyword rules classify that call
  and it's marked `source = "rules"`.
- **Explainable.** Every label carries a rationale a reviewer can check.
- **Business calls only.** Non-business calls (birthdays, farewells, socials) are screened out
  before classification (`01_explore`, section 4). That's why the taxonomy needs no *Social / other*
  theme: Claude never has to force a birthday call into a business category, and the themes'
  counts aren't padded by calls that aren't about work.

The full instructions sent to the model:

In [ ]:
print(SYSTEM_PROMPT)

In [ ]:
usage = load_usage()
topics_usage = usage[usage["step"] == "topics"]
print(f"Cached topic answers on disk: {len(list((PROCESSED_DIR / 'llm_cache').glob('topics_*.json')))}")
print(f"Topic API calls in the cost log: {len(topics_usage)}, total ${topics_usage['cost_usd'].sum():.4f} "
      f"(avg ${topics_usage['cost_usd'].mean():.4f} per call)" if len(topics_usage) else "No topic calls logged yet")
print(f"Cached answers from before cost logging existed: {uncosted_cache_entries()}")

## 6. Results: the categories

Counts below are for the **original 100 calls**.

In [ ]:
prim = Ho["primary_theme"].value_counts().rename("primary")
multi = Ho["themes"].map(list).explode().value_counts().rename("any mention")
tbl = pd.concat([prim, multi], axis=1).fillna(0).astype(int).reindex(THEME_NAMES)
tbl["share_primary"] = (tbl["primary"] / len(Ho)).round(2)
tbl.sort_values("any mention", ascending=False)

In [ ]:
long = tbl.reset_index(names="theme").melt(id_vars="theme", value_vars=["primary", "any mention"],
                                            var_name="count", value_name="calls")
fig = px.bar(long, y="theme", x="calls", color="count", barmode="group", orientation="h",
             category_orders={"theme": tbl.sort_values("any mention").index.tolist()[::-1]},
             color_discrete_map={"primary": "#1D4ED8", "any mention": "#93C5FD"},
             title="Calls per theme: primary theme vs any substantive mention (original 100)")
fig.update_layout(yaxis_title=None)
fig

In [ ]:
print("Themes per call:", Ho["themes"].map(len).value_counts().sort_index().to_dict())
print(f"Calls with 2+ themes: {(Ho['themes'].map(len) >= 2).mean():.0%}")

In [ ]:
heat = pd.crosstab(Ho["primary_theme"], Ho["call_type"]).reindex(THEME_NAMES).fillna(0).astype(int)
px.imshow(heat, text_auto=True, color_continuous_scale="Blues", aspect="auto",
          title="Primary theme by call type (original 100)").update_layout(coloraxis_showscale=False)

**What this shows.**

- **The Detect outage is the single largest theme** and appears in every call type: internal war
  rooms, customer escalations and support tickets. No other theme cuts across the company like that.
- **Compliance is the theme people talk about, not the one calls are about.** It appears in about
  half of all calls, but is rarely the *primary* theme. It's the context (customers are regulated)
  rather than the purpose of the call. This is why multi-label matters: a single-label classifier
  would make compliance look minor.
- **Most calls are about more than one thing.** That supports the decision to keep every
  substantive theme rather than force one label.
- **Each call type has a clear profile.** Renewals are external, Identity and Backup problems arrive
  through support, and planning is internal. That's expected, which gives confidence the labels are sensible.

## 7. Examples from each category

The assignment asks for examples. For each theme, the two most confident calls from the original
data, with Claude's one-line reasoning.

In [ ]:
ex = (Ho.sort_values("certainty", ascending=False)
        .groupby("primary_theme", group_keys=False).head(2)
        .assign(theme=lambda d: pd.Categorical(d["primary_theme"], THEME_NAMES, ordered=True))
        .sort_values(["theme", "certainty"], ascending=[True, False]))
ex[["theme", "call_type", "title", "certainty", "rationale"]].reset_index(drop=True)

In [ ]:
# Browse any theme in full: change THEME to any name in THEME_NAMES
THEME = "Billing disputes"
H[H["themes"].map(lambda t: THEME in t)][["date", "call_type", "title", "primary_theme", "certainty"]] \
  .sort_values("date").reset_index(drop=True)

## 8. Validation: should we trust the labels?

There's no ground truth in the data, so the labels are checked three ways.

### 8a. LLM vs keyword rules

The rules were written independently of the LLM, so agreement is evidence that both are capturing
something real, and disagreement shows where judgement was needed.

In [ ]:
cmp = H[["call_id", "title", "primary_theme", "rationale"]].merge(
    R[["call_id", "primary_theme"]].rename(columns={"primary_theme": "rules_primary"}), on="call_id")
agree = (cmp["primary_theme"] == cmp["rules_primary"]).mean()
pd.Series({
    "primary theme: LLM = rules": f"{agree:.0%}",
    "LLM primary appears anywhere in the rule themes": f"{hyb_metrics['primary_in_rule_themes']:.0%}",
    "theme-set overlap, LLM vs rules (mean Jaccard)": hyb_metrics["mean_jaccard_llm_vs_rules"],
    "cluster purity, hybrid labels (weighted)": hyb_metrics["cluster_purity_weighted"],
    "cluster purity, rules labels (weighted)": rul_metrics["cluster_purity_weighted"],
}, name="value").to_frame()

In [ ]:
ct = pd.crosstab(cmp["primary_theme"], cmp["rules_primary"]).reindex(index=THEME_NAMES, columns=THEME_NAMES).fillna(0).astype(int)
px.imshow(ct, text_auto=True, color_continuous_scale="Blues", aspect="auto",
          labels={"x": "rules primary", "y": "LLM primary"},
          title="Where the LLM and the rules agree (diagonal) and disagree").update_layout(coloraxis_showscale=False)

In [ ]:
dis = cmp[cmp["primary_theme"] != cmp["rules_primary"]]
print(f"{len(dis)} disagreements. Most common:")
print(dis.groupby(["rules_primary", "primary_theme"]).size().sort_values(ascending=False).head(5).to_string())
dis[["title", "rules_primary", "primary_theme", "rationale"]].head(8).reset_index(drop=True)

**What this shows.** The LLM and the rules pick the same primary theme for about three calls in
four, and the LLM's choice is almost always among the themes the rules also detected. So the LLM
is rarely inventing anything, and is mostly **choosing which of several plausible themes matters most**.

The disagreements are where that judgement adds value. The largest group is calls about the
**Comply v2 product launch** that the rules file under **Compliance & audit**, because the word
"compliance" appears everywhere. The LLM separates the *product* from the *regulatory work*.
Similarly, renewal calls dominated by the outage are labelled by what drove the conversation, not
by the word "renewal".

### 8b. Certainty: where is the model unsure?

In [ ]:
px.histogram(H, x="certainty", nbins=12, color="call_type", color_discrete_map=TYPE_COLORS,
             title="LLM certainty in the primary theme").update_layout(bargap=0.05)

In [ ]:
H.sort_values("certainty")[["certainty", "primary_theme", "themes", "title", "rationale"]].head(6).reset_index(drop=True)

**What this shows.** Certainty is high for most calls. The least certain calls are informative:

- **Genuinely mixed calls.** For example, a roadmap review dominated by the outage, or a renewal
  call where the outage, a competitor and pricing all matter. The multi-label `themes` field
  captures these properly even when the primary is a close call.
- **A gap in the taxonomy.** A LogVault SIEM-connector support case doesn't fit any theme well, and
  the model says so in its rationale. If integration issues grow, *Integrations / LogVault* should
  become an 11th theme. Low-certainty calls are a built-in signal for when to revise the taxonomy.

In production, calls below a certainty threshold (say 0.6) would go to a human review queue.

### 8c. Hand-labelled spot check

Agreement with rules isn't the same as being *right*. The final check is a human label on a sample.
The cell below picks a stratified sample of 20 original calls (two per predicted theme). If no
labels exist yet, it writes a template. Fill in `my_label` with one theme per call (without looking
at the predictions), save it as `data/labels/topic_gold.csv`, and re-run the cell to get accuracy for
both modes.

In [ ]:
LABEL_DIR = ROOT / "data" / "labels"
GOLD = LABEL_DIR / "topic_gold.csv"

if GOLD.exists():
    gold = pd.read_csv(GOLD).dropna(subset=["my_label"])
    g = gold.merge(H[["call_id", "primary_theme", "themes"]], on="call_id") \
            .merge(R[["call_id", "primary_theme"]].rename(columns={"primary_theme": "rules_primary"}), on="call_id")
    print(f"{len(g)} hand-labelled calls")
    print(f"  Hybrid primary = my label: {(g['primary_theme'] == g['my_label']).mean():.0%}")
    print(f"  Hybrid themes include my label: {g.apply(lambda r: r['my_label'] in set(r['themes']), axis=1).mean():.0%}")
    print(f"  Rules primary = my label:  {(g['rules_primary'] == g['my_label']).mean():.0%}")
    display(g[g["primary_theme"] != g["my_label"]][["title", "my_label", "primary_theme", "rules_primary"]])
else:
    sample = (Ho.sample(frac=1, random_state=0)         # shuffle, then take two per predicted theme
                .groupby("primary_theme").head(2)
                .sample(frac=1, random_state=1))          # reshuffle so the order doesn't hint at the answer
    LABEL_DIR.mkdir(parents=True, exist_ok=True)
    tmpl = LABEL_DIR / "topic_gold_template.csv"
    sample[["call_id", "call_type", "title"]].assign(my_label="").to_csv(tmpl, index=False)
    print(f"No labels yet. Wrote a {len(sample)}-call template to {tmpl.relative_to(ROOT)}")
    print("Valid labels:", "; ".join(THEME_NAMES))

## 9. Themes over time

Because each call is dated, themes can be trended. That's the bridge from *what* calls are about
to *how people feel* (`03_sentiment`).

In [ ]:
tp = (Ho.assign(period=pd.Categorical(Ho["period"], PERIODS, ordered=True))
        .explode("themes").groupby(["period", "themes"], observed=False).size().rename("calls").reset_index())
tp["share"] = tp["calls"] / tp["period"].map(Ho["period"].value_counts())
px.line(tp, x="period", y="share", color="themes", markers=True,
        title="Share of calls mentioning each theme, by period (original 100)",
        labels={"share": "share of calls in period", "themes": "theme"}).update_yaxes(tickformat=".0%")

In [ ]:
(Ho.groupby("primary_theme")
   .agg(calls=("call_id", "count"), avg_sentiment=("sentiment_score", "mean"))
   .sort_values("avg_sentiment").round(2))

**What this shows.** The themes follow the company's story, with two surprises.

- **The outage outlasts the incident.** Every call during the outage window touches it, as
  expected. But a month after full restoration it still comes up in more than half of all calls,
  *more* than during recovery, as renewals, audits and competitive evaluations bring it back up. For
  leadership, the cost of an incident is measured in months of conversations, not hours of downtime.
- **Competitive threat is outage-driven.** Competitor talk jumps from under 1 in 10 calls before
  the outage to about half of calls during it, then falls steadily after Comply v2 ships. That's
  consistent with the launch working as the "forward momentum" story the go-to-market team planned.
- **Comply v2** climbs during recovery (launch preparation) and stays high after GA.

Ranking themes by average sentiment shows the outage, Identity problems and competitive pressure
as the most painful subjects. `03_sentiment` tests this with our own transcript-level scoring
rather than relying on the vendor's score.

## 10. Does the pipeline handle new calls?

The synthetic transcripts were never seen when the taxonomy was designed. If they land in sensible
themes, the pipeline generalises beyond the calls it was built on.

In [ ]:
H[H["is_synthetic"]].sort_values("date")[["date", "call_type", "title", "primary_theme", "themes", "certainty"]] \
  .reset_index(drop=True)

## 11. Summary

| Question | Answer |
|---|---|
| **Categories** | 10 themes, from Detect outage & fallout through Billing disputes, each with a definition and a keyword rule |
| **Approach** | Hybrid. Clustering discovers, the taxonomy defines, Claude classifies (multi-label + certainty + rationale), rules provide a fallback and a check |
| **Why** | Provided topics are too fragmented; clustering alone forms groups around customer names and can't multi-label; rules alone match words, not meaning |
| **How much to trust it** | High agreement with independent rules; disagreements are explainable judgement calls; low-certainty calls flag taxonomy gaps; hand-label check in 8c |
| **Cost** | Fractions of a cent per call, cached so re-runs are free |
| **Generalises** | New (synthetic) calls are classified sensibly without changes |

**Limitations.** The taxonomy was designed on this dataset and would need periodic review (8b
shows how to spot when). Certainty is the model's own estimate, not a calibrated probability. The
hand-labelled sample is small.

**Next:** `03_sentiment.ipynb`